# Experiment No. 7
## Comparison of Different Regression Algorithms for Predictive Modeling

### Aim

To compare the performance of different regression algorithms on a given dataset and identify the most suitable algorithm for accurate prediction.

### Objectives

1. To understand the working principles of various regression algorithms.
2. To apply multiple regression models to the same dataset.
3. To evaluate model performance using suitable error metrics.
4. To determine which algorithm best fits the given problem based on accuracy and efficiency.

### Relevance

Regression analysis is a key predictive technique used in Machine Learning to estimate continuous outcomes, such as predicting house prices, temperature, or sales. Comparing multiple regression algorithms helps in selecting the most effective model for real-world applications, depending on data characteristics such as linearity, noise, and feature correlation.

### Step 1: Import Required Libraries

The required libraries are imported for numerical operations, data handling, splitting the dataset, feature transformation, regression models, and model evaluation.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### Step 2: Load Dataset

The California Housing dataset is loaded using Scikit-learn. It contains information about housing characteristics and a continuous target value representing the housing price.

In [2]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()

X, y = data.data, data.target

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (20640, 8)
Target shape: (20640,)


The feature names are displayed to understand the different variables used for predicting the target value.

In [3]:
print("Feature Names:")
print(data.feature_names)

Feature Names:
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']


### Step 3: Split Dataset

The dataset is divided into 80% training data and 20% testing data. The training data is used to train the regression models, while the testing data is used to evaluate their performance.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (16512, 8)
Testing data: (4128, 8)


### Step 4: Feature Scaling

Feature scaling is required for Support Vector Regression (SVR) because SVR is sensitive to the scale of the features. The target variable is also standardized for SVR.

In [5]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

y_train_scaled = scaler_y.fit_transform(
    y_train.reshape(-1, 1)
).ravel()

### Step 5: Initialize Regression Models

Five regression algorithms are created for comparison: Linear Regression, Polynomial Regression, Decision Tree Regression, Random Forest Regression, and Support Vector Regression.

In [6]:
models = {
    "Linear Regression": LinearRegression(),
    "Polynomial Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),
    "SVR": SVR(kernel='rbf')
}

models

{'Linear Regression': LinearRegression(),
 'Polynomial Regression': LinearRegression(),
 'Decision Tree': DecisionTreeRegressor(random_state=42),
 'Random Forest': RandomForestRegressor(random_state=42),
 'SVR': SVR()}

### Step 6: Polynomial Feature Transformation

Polynomial Regression extends Linear Regression by creating additional polynomial combinations of the original features. Here, degree 2 is used.

In [7]:
poly = PolynomialFeatures(degree=2)

X_poly_train = poly.fit_transform(X_train)
X_poly_test = poly.transform(X_test)

print("Original number of features:", X_train.shape[1])
print("Polynomial features:", X_poly_train.shape[1])

Original number of features: 8
Polynomial features: 45


### Step 7: Train and Evaluate the Models

Each regression model is trained using the training data and evaluated on the test data. MAE, MSE, and R² Score are calculated for every model.

For Polynomial Regression, polynomial features are used. For SVR, scaled features and scaled target values are used. The other models use the original features.

In [8]:
results = {}

for name, model in models.items():

    if name == "Polynomial Regression":
        model.fit(X_poly_train, y_train)
        y_pred = model.predict(X_poly_test)

    elif name == "SVR":
        model.fit(X_train_scaled, y_train_scaled)

        y_pred_scaled = model.predict(X_test_scaled)

        y_pred = scaler_y.inverse_transform(
            y_pred_scaled.reshape(-1, 1)
        ).ravel()

    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    results[name] = {
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred),
        "R2_Score": r2_score(y_test, y_pred)
    }

### Step 8: Display Results

The evaluation results of all five regression models are converted into a DataFrame so that their performance can be compared easily.

In [9]:
results_df = pd.DataFrame(results).T

print(results_df)

                            MAE       MSE  R2_Score
Linear Regression      0.533200  0.555892  0.575788
Polynomial Regression  0.516183  0.604538  0.538664
Decision Tree          0.454171  0.494256  0.622823
Random Forest          0.327731  0.256022  0.804624
SVR                    0.397283  0.354151  0.729740


### Step 9: Identify the Best Model

For regression, a lower MAE and MSE indicate lower prediction error, while a higher R² Score indicates that the model explains more variation in the target variable.

In [10]:
best_model = results_df["R2_Score"].idxmax()

print("Best Model based on R² Score:", best_model)

Best Model based on R² Score: Random Forest


This sorts the models according to their R² Score so that the models can be compared from the highest-performing to the lowest-performing model.

In [11]:
comparison = results_df.sort_values(
    by="R2_Score",
    ascending=False
)

print(comparison)

                            MAE       MSE  R2_Score
Random Forest          0.327731  0.256022  0.804624
SVR                    0.397283  0.354151  0.729740
Decision Tree          0.454171  0.494256  0.622823
Linear Regression      0.533200  0.555892  0.575788
Polynomial Regression  0.516183  0.604538  0.538664


## Conclusion

In this experiment, different regression algorithms were implemented and compared using the California Housing dataset. Linear Regression was used to model linear relationships, while Polynomial Regression was used to capture nonlinear relationships. Decision Tree Regression and Random Forest Regression were used to model more complex relationships, while Support Vector Regression was implemented using scaled data.

The models were evaluated using Mean Absolute Error (MAE), Mean Squared Error (MSE), and R² Score. A lower MAE and MSE indicate lower prediction error, while a higher R² Score indicates better model performance. Based on the obtained results, the model with the highest R² Score can be considered the most suitable model for this dataset.